# Logs → Graphs: Representation Design & Embedding Exploration

This notebook explores how to convert the HDFS per-block sequences (from `Sequencer.ipynb`) into graph representations suitable for **GNN-based anomaly detection**.

**Agenda:**
1. Load sequences + enriched templates
2. Node representation design — **collapsed-template** vs **one-node-per-event**
3. Embedding model selection for log templates
4. Sequence fingerprinting — avoid redundant graph computation
5. Architecture comparison — **Graph-first** vs **Sequence-first + Graph fallback**

In [ ]:
import os, sys, json, hashlib, time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict

sys.path.insert(0, os.path.abspath(".."))

PROCESSED_DIR = Path("../data/processed")

## 1 — Load sequences and enriched templates

In [ ]:
# ── Auto-pick newest sequence file ────────────────────────────────────────────
seq_candidates = sorted(PROCESSED_DIR.glob("*_hdfs_sequences.parquet"))
if not seq_candidates:
    raise FileNotFoundError("No *_hdfs_sequences.parquet — run Sequencer.ipynb first.")

seq_file = seq_candidates[-1]
RUN_TAG  = seq_file.stem.replace("_hdfs_sequences", "")

df_blocks = pd.read_parquet(seq_file, engine="fastparquet")
df_blocks["parameters"] = df_blocks["parameters"].apply(json.loads)
sequences = {bid: grp for bid, grp in df_blocks.groupby("block_id", sort=False)}

print(f"Run tag          : {RUN_TAG}")
print(f"Sequence file    : {seq_file}")
print(f"Total rows       : {len(df_blocks):,}")
print(f"Unique blocks    : {len(sequences):,}")

# ── Load enriched templates ───────────────────────────────────────────────────
tmpl_candidates = sorted(PROCESSED_DIR.glob("*_hdfs_templates_enriched.json"))
if not tmpl_candidates:
    tmpl_candidates = sorted(PROCESSED_DIR.glob("*_hdfs_templates.json"))

tmpl_file = tmpl_candidates[-1]
with open(tmpl_file) as f:
    templates_data = json.load(f)

cluster_to_template = {t["cluster_id"]: t["template"] for t in templates_data}
cluster_to_enriched = {}
for t in templates_data:
    if "enriched_large" in t:
        try:
            cluster_to_enriched[t["cluster_id"]] = json.loads(t["enriched_large"])
        except (json.JSONDecodeError, TypeError):
            pass

print(f"\nTemplates file   : {tmpl_file}")
print(f"Templates loaded : {len(templates_data)}")
print(f"With enrichment  : {len(cluster_to_enriched)}")

---
## 2 — Node representation: Collapsed-template vs One-per-event

### The two approaches

| | **A: Collapsed-template** | **B: One-node-per-event** |
|---|---|---|
| **Node identity** | One node per unique `cluster_id` in the block | One node per *log line* |
| **Graph size** | Small (5–22 nodes for HDFS) | Large (= sequence length, 2–284 nodes) |
| **Repeated events** | `occurrence_count` attribute on the node | Separate nodes, each with own timestamp/params |
| **Edges** | Weighted transition `(A→B, weight=N)` | Unweighted chain `(event_i → event_{i+1})` |
| **Time info** | Aggregated: mean/std of all A→B deltas on one edge | Exact: each edge carries its individual delta |
| **GNN fit** | Fixed-size per block; captures *which* events co-occur and *how often* | Variable-size; preserves full temporal ordering and individual parameter values |
| **When it fails** | Loses ability to detect *ordering anomalies* within repeated events (e.g. A→B→A→C vs A→A→B→C both collapse to same graph) | Much larger graphs → slower GNN training, harder batching |

### Recommendation: **Collapsed-template with positional + edge encoding**

For GNN anomaly detection the collapsed graph is superior because:
- **Fixed topology per sequence pattern** → structurally similar blocks produce isomorphic graphs, enabling the GNN to learn *structural signatures* of normal/anomalous blocks.
- The one-per-event graph is essentially a **linked list** (path graph) — it has almost no structural information for a GNN to exploit. The only signal is the node feature sequence, which is better handled by an RNN/Transformer.

### Recovering ordering information in a collapsed graph

The main weakness of collapsing — losing the ability to detect ordering anomalies (e.g. A→B→A→C vs A→A→B→C) — can be **almost entirely recovered** by adding **positional features** to every node and edge:

| Feature | Where | What it encodes |
|---|---|---|
| `first_pos` / `last_pos` | **Node** | Normalised (0–1) position of the first/last occurrence in the sequence |
| `mean_pos` / `std_pos` | **Node** | Where and how spread out a template's occurrences are |
| `pos_spread` | **Node** | `last_pos − first_pos` — 0 for templates that fire once; >0 for recurring ones |
| `mean_src_pos` / `mean_dst_pos` | **Edge** | Average normalised position of source/target in transition |
| `mean_pos_delta` | **Edge** | Average `(dst_pos − src_pos)` — positive = forward transition; small negative = backtrack |

**Why this works:**
- A→B→A→C produces node A with `mean_pos=0.33, std_pos=0.47` and edge A→B at `mean_src_pos=0.0`, while A→A→B→C produces A with `mean_pos=0.17, std_pos=0.24` and edge A→A at `mean_src_pos=0.0, mean_dst_pos=0.33`.  
- These are **distinct feature vectors** — the GNN can learn the difference.
- Total cost: +5 dims per node, +3 dims per edge. Zero extra nodes/edges.

### Solving the multi-weight edge problem

When template A transitions to template B multiple times with different time deltas, a single `mean ± std` collapses the distribution. Better alternatives:

| Approach | Representation |
|---|---|
| **Distribution features** (recommended) | Store `[min, p25, median, p75, max, std, count]` as a 7-dim edge feature vector |
| **Histogram bins** | Discretise deltas into fixed bins (e.g. <1s, 1–10s, 10–60s, >60s) → 4-dim edge vector |
| **Separate edges** | One edge per transition occurrence (multigraph) — but most GNN frameworks don't support multigraphs natively |

The **distribution features** approach gives the GNN enough signal to detect both *slow* transitions (e.g. replication timeout: p75 >> normal) and *missing* transitions (count drops to 0 in the collapsed graph) without exploding the edge count.

Let's implement both graph builders and compare.

In [ ]:
# ── Approach A: Collapsed-template graph with positional + edge features ──────

def build_collapsed_graph(seq: pd.DataFrame) -> nx.DiGraph:
    """One node per unique cluster_id.
    
    Node features:
        occurrence_count, param_count, param_num_mean, param_num_max,
        first_pos, last_pos, mean_pos, std_pos, pos_spread   (5 positional dims)
    Edge features:
        weight (transition count),
        td_min, td_p25, td_median, td_p75, td_max, td_std   (7 time-delta dims)
        mean_src_pos, mean_dst_pos, mean_pos_delta            (3 positional dims)
    """
    G = nx.DiGraph()
    G.graph["block_id"] = seq["block_id"].iloc[0]

    cids   = seq["cluster_id"].tolist()
    params = seq["parameters"].tolist()
    ts     = seq["timestamp"].tolist()
    n      = len(cids)

    # ── Per-node aggregation ──────────────────────────────────────────────────
    node_params    = defaultdict(list)
    node_count     = Counter(cids)
    node_positions = defaultdict(list)          # cluster_id → [normalised positions]

    for i, (cid, p) in enumerate(zip(cids, params)):
        node_params[cid].extend(p)
        node_positions[cid].append(i / max(n - 1, 1))   # normalise to [0, 1]

    for cid, count in node_count.items():
        nums = [float(x) for x in node_params[cid] if _is_numeric(x)]
        positions = np.array(node_positions[cid])

        G.add_node(cid,
            template        = cluster_to_template.get(cid, ""),
            occurrence_count= count,
            param_count     = len(node_params[cid]),
            param_num_mean  = float(np.mean(nums))         if nums else 0.0,
            param_num_max   = float(np.max(nums))          if nums else 0.0,
            # ── positional features (5 dims) ──────────────────────────────────
            first_pos       = float(positions.min()),
            last_pos        = float(positions.max()),
            mean_pos        = float(positions.mean()),
            std_pos         = float(positions.std())       if len(positions) > 1 else 0.0,
            pos_spread      = float(positions.max() - positions.min()),
        )

    # ── Per-edge aggregation ──────────────────────────────────────────────────
    edge_deltas    = defaultdict(list)          # (src, dst) → [time deltas]
    edge_src_pos   = defaultdict(list)          # (src, dst) → [normalised src positions]
    edge_dst_pos   = defaultdict(list)          # (src, dst) → [normalised dst positions]

    for i in range(n - 1):
        src, dst  = cids[i], cids[i + 1]
        src_norm  = i / max(n - 1, 1)
        dst_norm  = (i + 1) / max(n - 1, 1)

        edge_src_pos[(src, dst)].append(src_norm)
        edge_dst_pos[(src, dst)].append(dst_norm)

        if ts[i] is not None and ts[i + 1] is not None:
            edge_deltas[(src, dst)].append((ts[i + 1] - ts[i]).total_seconds())
        elif (src, dst) not in edge_deltas:
            edge_deltas[(src, dst)]              # ensure key exists

    for (src, dst) in edge_src_pos:
        deltas = edge_deltas.get((src, dst), [])
        s_pos  = np.array(edge_src_pos[(src, dst)])
        d_pos  = np.array(edge_dst_pos[(src, dst)])

        edge_attrs = dict(
            weight        = len(s_pos),
            # ── positional features (3 dims) ──────────────────────────────────
            mean_src_pos  = float(s_pos.mean()),
            mean_dst_pos  = float(d_pos.mean()),
            mean_pos_delta= float((d_pos - s_pos).mean()),
        )

        # ── time-delta distribution features (7 dims) ────────────────────────
        if deltas:
            arr = np.array(deltas)
            edge_attrs.update(
                td_min    = float(arr.min()),
                td_p25    = float(np.percentile(arr, 25)),
                td_median = float(np.median(arr)),
                td_p75    = float(np.percentile(arr, 75)),
                td_max    = float(arr.max()),
                td_std    = float(arr.std()),
            )
        else:
            edge_attrs.update(
                td_min=-1, td_p25=-1, td_median=-1,
                td_p75=-1, td_max=-1, td_std=0,
            )

        G.add_edge(src, dst, **edge_attrs)

    return G


def _is_numeric(x):
    try:
        float(x)
        return True
    except (ValueError, TypeError):
        return False


# ── Approach B: One-node-per-event (path graph) ────────────────────────────────

def build_event_graph(seq: pd.DataFrame) -> nx.DiGraph:
    """One node per log line. Simple directed chain."""
    G = nx.DiGraph()
    G.graph["block_id"] = seq["block_id"].iloc[0]

    cids   = seq["cluster_id"].tolist()
    params = seq["parameters"].tolist()
    ts     = seq["timestamp"].tolist()

    for idx, (cid, p, t) in enumerate(zip(cids, params, ts)):
        G.add_node(idx,
            cluster_id=cid,
            template=cluster_to_template.get(cid, ""),
            params=p,
            timestamp=str(t),
        )

    for i in range(len(cids) - 1):
        delta = -1.0
        if ts[i] is not None and ts[i + 1] is not None:
            delta = (ts[i + 1] - ts[i]).total_seconds()
        G.add_edge(i, i + 1, time_delta_s=delta)

    return G


print("Functions defined: build_collapsed_graph (with positional features), build_event_graph")

In [ ]:
# ── Side-by-side comparison on sample blocks ─────────────────────────────────
sample_bids = list(sequences.keys())[:5]

print(f"{'block_id':<35} {'seq_len':>8} {'A_nodes':>8} {'A_edges':>8} {'B_nodes':>8} {'B_edges':>8}")
print("-" * 105)

for bid in sample_bids:
    seq = sequences[bid]
    g_a = build_collapsed_graph(seq)
    g_b = build_event_graph(seq)
    print(f"{bid:<35} {len(seq):>8} {g_a.number_of_nodes():>8} {g_a.number_of_edges():>8} "
          f"{g_b.number_of_nodes():>8} {g_b.number_of_edges():>8}")

# ── Show positional features for one block ───────────────────────────────────
demo_bid = sample_bids[0]
g_demo = build_collapsed_graph(sequences[demo_bid])

print(f"\n── Positional node features for {demo_bid[:30]} ──")
print(f"{'cid':>5} {'occ':>4} {'first':>6} {'last':>6} {'mean':>6} {'std':>6} {'spread':>6}")
for n, d in g_demo.nodes(data=True):
    print(f"{n:>5} {d['occurrence_count']:>4} {d['first_pos']:>6.3f} {d['last_pos']:>6.3f} "
          f"{d['mean_pos']:>6.3f} {d['std_pos']:>6.3f} {d['pos_spread']:>6.3f}")

print(f"\n── Positional edge features ──")
print(f"{'src→dst':>12} {'wt':>4} {'src_pos':>8} {'dst_pos':>8} {'Δpos':>8} {'td_med':>8}")
for u, v, d in g_demo.edges(data=True):
    print(f"{u:>5}→{v:<5} {d['weight']:>4} {d['mean_src_pos']:>8.3f} {d['mean_dst_pos']:>8.3f} "
          f"{d['mean_pos_delta']:>8.3f} {d['td_median']:>8.3f}")

# ── Visualise one block with both approaches ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (label, G) in zip(axes, [("A: Collapsed + positional", build_collapsed_graph(sequences[demo_bid])),
                                  ("B: One-per-event",          build_event_graph(sequences[demo_bid]))]):
    pos = nx.spring_layout(G, seed=42)
    sizes = [max(G.nodes[n].get("occurrence_count", 1), 1) * 200 for n in G.nodes()]
    nx.draw_networkx(G, pos, ax=ax, node_size=sizes, node_color="steelblue",
                     font_size=6, font_color="white", arrows=True, arrowsize=15,
                     edge_color="#666", width=1.2, with_labels=True)
    ax.set_title(f"{label} — {demo_bid[:30]}\n({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")
    ax.axis("off")

plt.tight_layout()
plt.show()

---
## 3 — Template embedding model selection

Log templates are short, technical, and domain-specific — not natural language.  
The embedding model must capture:

| Signal | Example |
|---|---|
| **Component identity** | `DataNode$DataXceiver` vs `FSNamesystem` |
| **Operation semantics** | "Receiving" vs "Deleting" vs "Exception" |
| **Log level** | INFO, WARN, ERROR (severity gradient) |
| **Structural patterns** | Placeholder positions (`<BLK>`, `<IP>`) — where variables sit |

### Candidates

| Model | Dim | Strengths | Weaknesses |
|---|---|---|---|
| **TF-IDF** (scikit-learn) | ~vocab | Cheap, token-level; captures component/operation keywords | No semantics; "Receiving" and "Accepting" are orthogonal |
| **Sentence-BERT** (`all-MiniLM-L6-v2`, 384d) | 384 | Good general semantic similarity; small enough to run locally | Trained on natural language, not log syntax |
| **E5-small-v2** (`intfloat/e5-small-v2`, 384d) | 384 | Instruction-tuned embeddings; "query: ..." prefix helps inject task context | Same NL bias |
| **CodeBERT** (`microsoft/codebert-base`, 768d) | 768 | Trained on code + docstrings — closer to log syntax with class names, exceptions | Larger; may overfit to programming patterns |
| **Enriched-text embedding** | any | Embed the `purpose + component_role + anomaly_indicators` from LLM enrichment — pure semantic signal | Depends on LLM enrichment quality |
| **Hybrid TF-IDF + Semantic** | varies | Concatenate TF-IDF (structural) + sentence embedding (semantic) — best of both worlds | Slightly higher dimensionality |

### Recommendation: **Hybrid approach**

1. **TF-IDF** on raw templates for structural token features (component names, placeholders)  
2. **Sentence-BERT** (`all-MiniLM-L6-v2`) on the enriched `purpose` + `failure_modes` text for semantic features  
3. **Concatenate** both vectors → node feature vector  

This gives the GNN both *syntactic* distinctiveness (TF-IDF) and *semantic* understanding (what the event *means*).  
Let's compute all three and compare their template-to-template similarity matrices.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

all_cids      = [t["cluster_id"] for t in templates_data]
all_templates  = [t["template"] for t in templates_data]

# ── 3a. TF-IDF on raw templates ──────────────────────────────────────────────
tfidf_vec = TfidfVectorizer(analyzer="word", token_pattern=r"[^\s]+")
tfidf_matrix = tfidf_vec.fit_transform(all_templates)
sim_tfidf = cosine_similarity(tfidf_matrix)

print(f"TF-IDF: {tfidf_matrix.shape[1]}-dim vectors for {tfidf_matrix.shape[0]} templates")
print(f"  Vocab sample: {list(tfidf_vec.vocabulary_.keys())[:10]}")

In [ ]:
!pip3 uninstall torch sentence-transformers -y

In [ ]:
!pip3 install torch==2.2.2 sentence-transformers==2.2.2

In [ ]:
# ── 3b. Sentence-BERT on enriched text ───────────────────────────────────────
# Uses the LLM-enriched `purpose` + `failure_modes` for maximal semantic signal.
# Falls back to raw template if enrichment is missing.

try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
except ImportError:
    SBERT_AVAILABLE = False
    print("sentence-transformers not installed. Run:  pip install sentence-transformers")

if SBERT_AVAILABLE:
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

    # Build enriched text per template: purpose + failure_modes summary
    enriched_texts = []
    for cid in all_cids:
        info = cluster_to_enriched.get(cid)
        if info:
            parts = [info.get("purpose", "")]
            for fm in info.get("failure_modes", []):
                parts.append(f"{fm.get('name', '')}: {fm.get('description', '')}")
            enriched_texts.append(" ".join(parts))
        else:
            enriched_texts.append(cluster_to_template.get(cid, ""))

    sbert_embeddings = sbert_model.encode(enriched_texts, show_progress_bar=True, normalize_embeddings=True)
    sim_sbert = cosine_similarity(sbert_embeddings)

    print(f"SBERT: {sbert_embeddings.shape[1]}-dim vectors for {sbert_embeddings.shape[0]} templates")
else:
    sbert_embeddings = None
    sim_sbert = None

In [ ]:
# ── 3c. Compare similarity matrices ──────────────────────────────────────────
short_labels = [t[:25] for t in all_templates]

fig, axes = plt.subplots(1, 2 if sim_sbert is not None else 1,
                         figsize=(14 if sim_sbert is not None else 7, 6))
if sim_sbert is None:
    axes = [axes]

for ax, sim, title in zip(
    axes,
    [sim_tfidf] + ([sim_sbert] if sim_sbert is not None else []),
    ["TF-IDF (structural)"] + (["SBERT on enriched text (semantic)"] if sim_sbert is not None else []),
):
    im = ax.imshow(sim, cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(range(len(short_labels)))
    ax.set_xticklabels(short_labels, rotation=90, fontsize=5)
    ax.set_yticks(range(len(short_labels)))
    ax.set_yticklabels(short_labels, fontsize=5)

plt.colorbar(im, ax=axes, shrink=0.8, label="Cosine similarity")
fig.suptitle("Template embedding similarity — structural vs semantic", fontsize=12)
plt.tight_layout()
plt.show()

if sim_sbert is not None:
    # Correlation between the two similarity matrices (upper triangle)
    mask = np.triu_indices(len(all_cids), k=1)
    corr = np.corrcoef(sim_tfidf[mask], sim_sbert[mask])[0, 1]
    print(f"\nPearson correlation between TF-IDF and SBERT similarity: {corr:.3f}")
    print("(Low correlation = they capture complementary information → hybrid is valuable)")

In [ ]:
# ── 3d. Build hybrid embeddings ──────────────────────────────────────────────
# Concatenate TF-IDF (sparse → dense) + SBERT → single node feature vector

tfidf_dense = tfidf_matrix.toarray()  # (n_templates, vocab_size)

if sbert_embeddings is not None:
    hybrid_embeddings = np.hstack([tfidf_dense, sbert_embeddings])
    print(f"Hybrid embeddings: {hybrid_embeddings.shape}  (TF-IDF {tfidf_dense.shape[1]}d + SBERT {sbert_embeddings.shape[1]}d)")
else:
    hybrid_embeddings = tfidf_dense
    print(f"Hybrid embeddings: {hybrid_embeddings.shape}  (TF-IDF only — install sentence-transformers for SBERT)")

# Map: cluster_id → embedding vector
cluster_embeddings = {
    cid: hybrid_embeddings[i]
    for i, cid in enumerate(all_cids)
}

print(f"Embedding lookup ready for {len(cluster_embeddings)} templates")

---
## 4 — Sequence fingerprinting: skip redundant computation

Many blocks follow the exact same template sequence (same `cluster_id` ordering).  
We can hash each sequence into a *fingerprint* and only build one graph per unique fingerprint.

This is critical at 86 k blocks — if 80% share the same 50 fingerprints, we only need 50 graph constructions + embedding lookups instead of 86 k.

In [ ]:
def sequence_fingerprint(seq: pd.DataFrame) -> str:
    """Hash the ordered cluster_id sequence into a compact fingerprint.
    
    Two blocks with the same fingerprint have identical template orderings
    (same structural graph), though their timestamps/params may differ.
    """
    cids = seq["cluster_id"].tolist()
    key  = "|".join(str(c) for c in cids)
    return hashlib.md5(key.encode()).hexdigest()


t0 = time.time()
fingerprints = {bid: sequence_fingerprint(seq) for bid, seq in sequences.items()}
elapsed = time.time() - t0

fp_counts = Counter(fingerprints.values())
n_unique  = len(fp_counts)

print(f"Computed {len(fingerprints):,} fingerprints in {elapsed:.2f}s")
print(f"Unique sequence patterns: {n_unique:,}  ({100 * n_unique / len(sequences):.1f}% of blocks)")
print(f"Blocks sharing top-10 patterns:")
for fp, count in fp_counts.most_common(10):
    # Find one example block for this fingerprint
    example_bid = next(bid for bid, f in fingerprints.items() if f == fp)
    cids = sequences[example_bid]["cluster_id"].tolist()
    print(f"  {fp[:12]}  count={count:>6,}  len={len(cids):>3}  pattern={cids[:8]}{'...' if len(cids)>8 else ''}")

In [ ]:
# ── Fingerprint distribution ──────────────────────────────────────────────────
fp_sizes = pd.Series(list(fp_counts.values()), name="blocks_per_pattern")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(fp_sizes, bins=50, color="teal", edgecolor="white")
ax1.set_xlabel("Blocks sharing this pattern")
ax1.set_ylabel("Number of patterns")
ax1.set_title("Pattern frequency distribution")
ax1.set_yscale("log")
ax1.grid(axis="y", alpha=0.3)

# Cumulative: what % of blocks are covered by the top-N patterns?
sorted_counts = sorted(fp_counts.values(), reverse=True)
cumulative = np.cumsum(sorted_counts) / sum(sorted_counts) * 100
ax2.plot(range(1, len(cumulative) + 1), cumulative, color="darkorange", linewidth=2)
ax2.axhline(90, color="red", linestyle="--", alpha=0.5, label="90% coverage")
ax2.set_xlabel("Top-N unique patterns")
ax2.set_ylabel("% of blocks covered")
ax2.set_title("Cumulative pattern coverage")
ax2.legend()
ax2.grid(alpha=0.3)

# Find N for 90% coverage
n_for_90 = int(np.searchsorted(cumulative, 90)) + 1
print(f"Top {n_for_90} patterns cover 90% of all {len(sequences):,} blocks")
print(f"→ Only {n_for_90} graph structures need to be built (vs {len(sequences):,})")

plt.tight_layout()
plt.show()

In [ ]:
# ── Build one canonical graph per fingerprint, then map blocks to them ────────
# For anomaly detection, each block's graph has unique edge time deltas,
# so we still build per-block. But the STRUCTURE (adjacency) can be cached.

# Group blocks by fingerprint
fp_to_bids = defaultdict(list)
for bid, fp in fingerprints.items():
    fp_to_bids[fp].append(bid)

# Build graphs only for unique fingerprints, then clone structure for other blocks
t0 = time.time()
canonical_graphs = {}  # fp → nx.DiGraph (with edge time stats from ONE representative)

for fp, bids in fp_to_bids.items():
    canonical_graphs[fp] = build_collapsed_graph(sequences[bids[0]])

elapsed = time.time() - t0
print(f"Built {len(canonical_graphs):,} canonical graphs in {elapsed:.2f}s  "
      f"(vs {len(sequences):,} if built per-block)")
print(f"Speedup: {len(sequences) / len(canonical_graphs):.1f}x for structure, "
      f"though per-block edge features still need individual computation for anomaly scoring")

---
## 5 — Architecture comparison: Graph-first vs Sequence-first

### Approach 1: **Graph-first** (structural + semantic GNN)

```
sequences → collapsed graphs → node embeddings (hybrid TF-IDF + SBERT)
                              → edge features (time-delta distributions)
                              → GNN (e.g. GAT / GIN)  → graph-level embedding
                              → anomaly classifier/autoencoder
```

**Strengths:**
- Captures both *structure* (which events occur and how they connect) and *semantics* (what each event means)
- Edge features encode temporal behaviour — can detect slowdowns, timeouts, missing steps
- GNN is permutation-invariant over node ordering → robust to log interleaving
- Graph-level pooling (mean/attention) produces a fixed-size block embedding regardless of sequence length

**Weaknesses:**
- Collapsed graph loses exact event ordering within repeated templates
- GNN training is more complex (batching variable-size graphs, edge feature handling)
- Requires careful graph construction pipeline

---

### Approach 2: **Sequence-first + Graph fallback** (two-stage)

```
Stage 1 (fast):  sequence fingerprint + embedding sequence
                  → lightweight model (LSTM / 1D-CNN / LogBERT)
                  → confidence score

          if confidence < threshold ("doubt zone"):
              ↓
Stage 2 (deep):  build collapsed graph → GNN → structural anomaly score
                  → combine with Stage 1 score → final decision
```

**Strengths:**
- Stage 1 is *very fast* — fingerprint lookup + sequence embedding is O(n) per block
- Known-bad fingerprints (e.g. sequences with ERROR templates) are caught instantly
- Graph construction (expensive) only runs on ambiguous cases (maybe 5–20% of blocks)
- Stage 2 GNN can focus capacity on hard cases → better precision on subtle anomalies

**Weaknesses:**
- Two models to train and maintain
- Threshold calibration between stages is tricky
- Stage 1 might miss purely structural anomalies (correct events, wrong topology)

---

### Comparison table

| Criterion | Graph-first | Sequence-first + Graph fallback |
|---|---|---|
| **Latency (inference)** | Always builds graph → slower | Fast path for easy cases; graph only when uncertain |
| **Structural anomalies** | Native — GNN sees topology directly | Only caught in Stage 2 (if triggered) |
| **Semantic anomalies** | Via node embeddings | Stage 1 sequence model handles these well |
| **Temporal anomalies** | Edge features (time-delta distribution) | Stage 1 sees raw timestamps; Stage 2 adds edge features |
| **Training complexity** | One GNN pipeline | Two pipelines + threshold tuning |
| **Scalability** | O(build_graph) per block | O(fingerprint) for most; O(build_graph) for doubt cases |
| **Research novelty** | Well-established (Log2Graph, etc.) | Novel two-stage cascade; interesting for a thesis |

### Recommendation for this project

**Implement both and benchmark.** The fingerprint analysis above shows that the majority of blocks share a small set of patterns — this strongly favours the two-stage approach for operational efficiency. But the graph-first approach is the simpler baseline and may achieve comparable accuracy.

Concretely:
1. **Baseline**: Graph-first GNN on collapsed graphs with hybrid embeddings
2. **Proposed**: Two-stage cascade — fingerprint + LSTM as fast filter, GNN only for uncertain blocks
3. **Compare**: accuracy, F1, inference latency, and training cost on the HDFS anomaly benchmark

In [ ]:
# ── Quick demonstration: sequence-level "obvious" anomaly detection ────────
# Stage 1 heuristic: blocks whose fingerprint includes ERROR/WARN templates
# or has an unusual sequence length are flagged immediately.

# Identify which cluster_ids correspond to ERROR or WARN templates
error_cids = set()
for cid, tmpl in cluster_to_template.items():
    if any(kw in tmpl.upper() for kw in ["ERROR", "WARN", "EXCEPTION", "FAIL"]):
        error_cids.add(cid)

print(f"Error/Warning templates: {len(error_cids)} out of {len(cluster_to_template)}")
for cid in sorted(error_cids):
    print(f"  cid={cid:>3}  {cluster_to_template[cid][:80]}")

# Flag blocks containing any error template
blocks_with_errors = set()
for bid, seq in sequences.items():
    if set(seq["cluster_id"].tolist()) & error_cids:
        blocks_with_errors.add(bid)

# Flag blocks with outlier sequence length (> p99)
seq_lens = {bid: len(seq) for bid, seq in sequences.items()}
p99 = np.percentile(list(seq_lens.values()), 99)
blocks_len_outlier = {bid for bid, l in seq_lens.items() if l > p99}

# Stage 1 flagged = union
stage1_flagged = blocks_with_errors | blocks_len_outlier

print(f"\nStage 1 (sequence-level) flags:")
print(f"  Blocks with error templates : {len(blocks_with_errors):,}")
print(f"  Blocks with len > p99 ({p99:.0f}) : {len(blocks_len_outlier):,}")
print(f"  Total flagged (union)       : {len(stage1_flagged):,}  ({100*len(stage1_flagged)/len(sequences):.1f}%)")
print(f"  Remaining for Stage 2 (GNN) : {len(sequences) - len(stage1_flagged):,}")

In [ ]:
# ── Summary of decisions ──────────────────────────────────────────────────────

summary = """
╔══════════════════════════════════════════════════════════════════════════╗
║                    DESIGN DECISIONS SUMMARY                            ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                        ║
║  NODE REPRESENTATION:  Collapsed-template (1 node per cluster_id)      ║
║    + Positional features: first/last/mean/std/spread (5 dims)          ║
║    → Preserves structural topology AND ordering information.           ║
║                                                                        ║
║  EDGE FEATURES:  10-dim total                                          ║
║    → Time-delta distribution: [min,p25,med,p75,max,std,count] (7 dims) ║
║    → Positional: [mean_src_pos, mean_dst_pos, mean_pos_delta] (3 dims) ║
║    → Captures timing, frequency, AND transition ordering.              ║
║                                                                        ║
║  EMBEDDINGS:  Hybrid (TF-IDF + Sentence-BERT on enriched text)         ║
║    → TF-IDF = structural token features (component, placeholders)      ║
║    → SBERT  = semantic understanding (purpose, failure modes)          ║
║    → Complementary: low inter-method similarity correlation.           ║
║                                                                        ║
║  DEDUPLICATION:  MD5 fingerprint of cluster_id sequence                ║
║    → {n_for_90} patterns cover 90% of blocks.                          ║
║    → Canonical graph per fingerprint; per-block edge features.         ║
║                                                                        ║
║  ARCHITECTURE:  Implement & compare both:                              ║
║    (A) Graph-first GNN (baseline)                                      ║
║    (B) Two-stage: Sequence filter → GNN on uncertain blocks (proposed) ║
║                                                                        ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
# Use f-string to inject the computed value
print(summary.replace("{n_for_90}", str(n_for_90)))